# Transfer learning for heart murmur detection

Compare four approaches on the same patients: the repo's **Random Forest**, its **CNN trained from scratch**, a **pretrained YAMNet CNN with a new murmur classifier**, and a **pretrained CardioPHON encoder with a new murmur classifier**. Both pretrained encoders stay frozen. Only the new classifier is trained. This compares complete approaches because YAMNet also changes how the audio is represented.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "config.py").is_file():
    raise FileNotFoundError("Open the notebook from the repo root or its notebooks folder.")
sys.path.insert(0, str(root))

FIGURES_DIR = root / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

split_file = root / "data" / "processed" / "recording_splits.csv"
outputs = root / "outputs"
for path in [split_file, outputs / "rf_window_predictions.csv",
             outputs / "cnn_window_predictions.csv"]:
    if not path.is_file():
        raise FileNotFoundError(path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from scipy.signal import resample_poly
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

plt.style.use("dark_background")

from src.config import LABELS
from src.preprocessing.audio_preprocessing import iter_recording_windows
from src.models.data_utils import SEED, SPLIT_NAMES, encode_labels, make_predictions_table
from src.evaluation.aggregate_patients import aggregate_patients
from src.evaluation.comparison import window_index, compare_models, plot_patient_roc

keras.utils.set_random_seed(SEED)
recordings = pd.read_csv(split_file, dtype={"patient_id": str})
split_by_patient = recordings.groupby("patient_id")[["split", "murmur_label"]]
if not split_by_patient.nunique(dropna=False).eq(1).all().all():
    raise ValueError("A patient has conflicting labels or belongs to multiple splits.")

baseline_predictions = {
    "Random Forest": pd.read_csv(outputs / "rf_window_predictions.csv", dtype={"subject_id": str}),
    "Original CNN": pd.read_csv(outputs / "cnn_window_predictions.csv", dtype={"subject_id": str}),
}

reference = window_index(baseline_predictions["Random Forest"])
pd.testing.assert_frame_equal(reference, window_index(baseline_predictions["Original CNN"]))

local_patients = recordings[["patient_id", "split", "murmur_label"]].drop_duplicates()
local_patients = local_patients.rename(columns={"patient_id": "subject_id", "murmur_label": "true_label"})
local_patients["split"] = local_patients["split"].map(SPLIT_NAMES)
expected = reference[["subject_id", "split", "true_label"]].drop_duplicates()
matched = expected.merge(local_patients, on=["subject_id", "split", "true_label"], how="left", indicator=True)
if not matched["_merge"].eq("both").all():
    raise ValueError("The local patient split or labels differ from the saved RF/CNN run.")

print(expected.groupby("split")["subject_id"].nunique())

## Extract frozen YAMNet features

Use the repo's same four-second windows. Resample the filtered mono audio from 4 kHz to YAMNet's expected 16 kHz. YAMNet returns one embedding per audio frame. Average the embeddings within each window so the new classifier gets one vector per window.

Check that the window IDs, labels, and splits match the saved RF and CNN runs before training.

In [ ]:
import tarfile

# To use an already extracted model, set this to the folder with saved_model.pb.
LOCAL_YAMNET = None
if LOCAL_YAMNET is None:
    archive = Path(keras.utils.get_file(
        fname="yamnet-v1.tar.gz",
        origin="https://tfhub.dev/google/yamnet/1?tf-hub-format=compressed",
    ))
    model_folder = archive.parent / "yamnet-v1"
    if not (model_folder / "saved_model.pb").is_file():
        try:
            with tarfile.open(archive, "r:gz") as model_archive:
                model_archive.extractall(model_folder, filter="data")
        except tarfile.ReadError as error:
            archive.unlink(missing_ok=True)
            raise RuntimeError(
                "The YAMNet download was not a model archive. "
                "Download and extract the YAMNet SavedModel manually, "
                "then set LOCAL_YAMNET to its folder."
            ) from error
else:
    model_folder = Path(LOCAL_YAMNET)

if not (model_folder / "saved_model.pb").is_file():
    raise FileNotFoundError(f"Expected saved_model.pb in {model_folder}")

yamnet = tf.saved_model.load(str(model_folder))

features, rows = [], []
for count, (window, sample_rate, meta) in enumerate(iter_recording_windows(recordings), start=1):
    waveform = resample_poly(window, 16000, sample_rate)
    waveform = np.clip(waveform, -1, 1).astype(np.float32)
    _, embeddings, _ = yamnet(tf.convert_to_tensor(waveform))
    features.append(tf.reduce_mean(embeddings, axis=0).numpy())
    rows.append({
        "subject_id": meta["patient_id"],
        "recording_id": Path(meta["recording"]).stem,
        "location": meta["location"],
        "window_idx": meta["window_idx"],
        "split": SPLIT_NAMES[meta["split"]],
        "true_label": meta["murmur_label"],
    })
    if count % 1000 == 0:
        print(f"Extracted {count:,} windows")

windows = pd.DataFrame(rows)
x = np.stack(features)
del features, rows
pd.testing.assert_frame_equal(reference, window_index(windows))
print("Matched windows:", len(windows))
print("Embedding shape:", x.shape)

## Train the new classifier

Fit input normalization and class weights on the training split. Use clean validation loss for early stopping. The held-out test labels are used only in the final comparison.

In [ ]:
def train_transfer_head(windows, features):
    """Fit the same small classifier to either encoder's embeddings."""
    train = windows["split"].eq("train").to_numpy()
    val = windows["split"].eq("val").to_numpy()
    y = encode_labels(windows["true_label"])
    classes = np.arange(len(LABELS))
    weights = compute_class_weight("balanced", classes=classes, y=y[train])

    normalize = keras.layers.Normalization()
    normalize.adapt(features[train])
    classifier = keras.Sequential([
        keras.Input(shape=(features.shape[1],)),
        normalize,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(len(LABELS), activation="softmax"),
    ])
    classifier.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    history = classifier.fit(
        features[train], y[train],
        validation_data=(features[val], y[val]),
        class_weight=dict(enumerate(weights)),
        epochs=40, batch_size=32, verbose=2,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=6, restore_best_weights=True
        )],
    )

    return classifier, history


classifier, history = train_transfer_head(windows, x)

for key, label in [("loss", "Training"), ("val_loss", "Validation")]:
    values = history.history[key]
    plt.plot(range(1, len(values) + 1), values, label=label)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig(FIGURES_DIR / "transfer_learning_loss.png", dpi=200, bbox_inches="tight")
plt.show()

## Extract CardioPHON embeddings and train a new head

Import the official `AudioNTT2022` encoder and load its 4 kHz pretrained weights strictly. The helper downloads the source and checkpoint from pinned upstream revision `a3bf9709`. It uses the same filtered four-second windows and checks their identities, splits, and labels against RF/CNN. The official frontend uses 64 Mel bands, a 512-sample FFT, a 160-sample hop, and 60–2000 Hz. Fit log-Mel normalization on training windows only and reuse it for validation/test.

Train the same 64-unit Keras head and early-stopping settings used for YAMNet. Class order is the project's **Present, Absent, Unknown**; the upstream classifier's different order is not reused. All patients remain included; demographic fusion and quality filtering are outside this audio-only comparison. See the [official model source](https://git.lih.lu/vdespotovic/cardiophon/-/tree/a3bf9709/heart_sound_classification).


In [ ]:
import json
from src.models.cardiophon import extract_embeddings

cardio_windows, cardio_x, cardio_normalization = extract_embeddings(
    recordings, root / "data/processed/cardiophon/a3bf9709", reference,
)
keras.utils.set_random_seed(SEED)
cardio_classifier, cardio_history = train_transfer_head(cardio_windows, cardio_x)
cardiophon_predictions = make_predictions_table(
    cardio_windows, cardio_classifier.predict(cardio_x, batch_size=128, verbose=0),
)
cardiophon_predictions.to_csv(outputs / "cardiophon_window_predictions.csv", index=False)
cardio_classifier.save(outputs / "cardiophon_classifier.keras")
pd.DataFrame(cardio_history.history).to_csv(outputs / "cardiophon_training_history.csv", index=False)
(outputs / "cardiophon_metadata.json").write_text(json.dumps({
    "frontend_normalization": cardio_normalization, "labels": LABELS,
    "seed": SEED, "window_seconds": 4, "quality_filtering": False,
    "demographics": False, "head": "Normalization-Dense64-Dropout0.3-Dense3",
}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for ax, (name, training_history) in zip(axes, [
    ("YAMNet", history.history), ("CardioPHON", cardio_history.history),
]):
    for key, label in [("loss", "Training"), ("val_loss", "Validation")]:
        values = training_history[key]
        ax.plot(range(1, len(values) + 1), values, label=label)
    ax.set(title=name + " classifier loss", xlabel="Epoch", ylabel="Loss")
    ax.legend()
fig.savefig(FIGURES_DIR / "transfer_learning_loss.png", dpi=200)
plt.show()
plt.close(fig)


## Compare the four models

Average window probabilities within each recording, then average across recordings to make one prediction per patient. Report validation separately from the held-out test comparison. **Present recall** tells us how many patients with the Present label the model caught. **Macro-F1** gives all three labels equal weight.

In [ ]:
transfer_probs = classifier.predict(x, batch_size=128, verbose=0)
predictions = {
    **baseline_predictions,
    "CardioPHON transfer CNN": cardiophon_predictions,
    "YAMNet transfer CNN": make_predictions_table(windows, transfer_probs),
}

predictions["YAMNet transfer CNN"].to_csv(outputs / "yamnet_window_predictions.csv", index=False)

validation, _ = compare_models(predictions, "val")
validation.to_csv(outputs / "transfer_learning_validation_comparison.csv")
print("Validation patients")
display(validation.round(3))

In [ ]:
test, test_patients = compare_models(predictions, "test")
test.to_csv(outputs / "transfer_learning_test_comparison.csv")
print("Held-out test patients")
display(test.round(3))

columns = ["Accuracy", "Macro-F1", "Present recall", "Macro ROC-AUC"]
ax = test[columns].T.plot.bar(rot=0, ylim=(0, 1.08), figsize=(13, 5))
ax.set(title="Held-out patient comparison", ylabel="Score")
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", padding=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "transfer_learning_comparison.png", dpi=200)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
axes = axes.ravel()
for ax, (name, patients) in zip(axes, test_patients.items()):
    ConfusionMatrixDisplay.from_predictions(
        patients["true_label"], patients["pred_label"],
        labels=LABELS, cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(name)
fig.savefig(FIGURES_DIR / "transfer_learning_confusion_matrices.png", dpi=200)
plt.show()

## CNN versus transfer learning ROC curves

Compare the original CNN with the frozen YAMNet and CardioPHON transfer approaches on the same held-out test patients. Each panel shows one-versus-rest ROC curves for Present, Absent, and Unknown after giving each recording equal weight.

In [ ]:
names = ["Original CNN", "YAMNet transfer CNN", "CardioPHON transfer CNN"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, names):
    plot_patient_roc(ax, test_patients[name], name + ": Patient-Level ROC")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cnn_vs_transfer_learning_roc_curves.png", dpi=300)
plt.show()
plt.close(fig)


## Interpret the comparison

Use the newly generated held-out table and ROC panels to compare all four approaches; no CardioPHON result is available until the cells above finish. Compare macro-F1, Present recall, and per-class AUC alongside accuracy. This experiment compares complete audio pipelines, including their different representations. It uses CardioPHON's frozen audio encoder with a new local head, without quality rejection or demographics, and does not reproduce the published full CardioPHON system. Its pretraining datasets exclude CirCor22 according to the [official release](https://git.lih.lu/vdespotovic/cardiophon).
